In [ ]:
# Step 1: Mount Google Drive and set up paths
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import shutil
import random
from collections import defaultdict
from tqdm import tqdm
import yaml

# Base directory – adjust if your folder is named differently
BASE_DIR = "/content/drive/MyDrive/final_yolo_retraining (1)"

# Paths to input data
COCO_IMAGES_ROOT = os.path.join(BASE_DIR, "coco/images")      # contains class subfolders
COCO_ANNOT_PATH = os.path.join(BASE_DIR, "coco/annotations/instances_train2017.json")
CUSTOM_ROOT = os.path.join(BASE_DIR, "custom")                # contains crosswalk, stairs, pothole folders

# Output paths – everything under BASE_DIR
OUTPUT_DATASET_DIR = os.path.join(BASE_DIR, "yolo_dataset")   # final YOLO dataset
TRAINING_OUTPUT_DIR = os.path.join(BASE_DIR, "runs")          # training results

# Create directories
os.makedirs(OUTPUT_DATASET_DIR, exist_ok=True)
os.makedirs(TRAINING_OUTPUT_DIR, exist_ok=True)
for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DATASET_DIR, 'labels', split), exist_ok=True)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.flush_and_unmount()

In [ ]:
if os.path.exists(BASE_DIR) and "final_yolo_retraining" in BASE_DIR and BASE_DIR != "/content/drive/MyDrive/final_yolo_retraining":
    print("Warning: BASE_DIR appears to be nested. Check your path!")
    # Optionally, exit or correct it

In [ ]:
# ------------------------------------------------------------
# Step 2: Parameters
# ------------------------------------------------------------
SELECTED_COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'train', 'truck',
    'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'backpack', 'umbrella', 'handbag', 'suitcase',
    'bottle', 'cup', 'knife', 'spoon', 'bowl', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop',
    'cell phone', 'sink', 'refrigerator', 'book', 'clock', 'vase',
    'scissors', 'toothbrush'
]

MAX_IMAGES_PER_COCO_CLASS = 600   # limit per class

CUSTOM_CLASSES = ['crosswalk', 'stairs', 'pothole']
# Original YOLO label IDs in your custom folders (assumed 0,1,2)
CUSTOM_ORIGINAL_IDS = {'crosswalk': 0, 'stairs': 1, 'pothole': 2}

EPOCHS = 100
BATCH_SIZE = 16
IMAGE_SIZE = 640
MODEL_SIZE = 'm'          # n, s, m, l, x
SAVE_PERIOD = 10          # save checkpoint every 10 epochs


In [ ]:
# ------------------------------------------------------------
# Step 3: Build file name to full path mapping for COCO images
# ------------------------------------------------------------
print("Scanning COCO images directory for files...")
filename_to_path = {}
for root, dirs, files in os.walk(COCO_IMAGES_ROOT):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            filename_to_path[file] = os.path.join(root, file)

print(f"Found {len(filename_to_path)} image files in COCO folders.")

# ------------------------------------------------------------
# Step 4: Load COCO annotations and build class mapping
# ------------------------------------------------------------
with open(COCO_ANNOT_PATH, 'r') as f:
    coco_data = json.load(f)

coco_cat_id_to_name = {cat['id']: cat['name'] for cat in coco_data['categories']}

# Find COCO ids for selected classes
selected_coco_ids = [cid for cid, name in coco_cat_id_to_name.items() if name in SELECTED_COCO_CLASSES]
print(f"Selected COCO class IDs: {selected_coco_ids}")

# Final class list: COCO classes first, then custom classes
final_classes = SELECTED_COCO_CLASSES + CUSTOM_CLASSES
class_id_map = {name: idx for idx, name in enumerate(final_classes)}
print("Final class list:", final_classes)


Scanning COCO images directory for files...
Found 10228 image files in COCO folders.
Selected COCO class IDs: [1, 2, 3, 4, 6, 7, 8, 10, 11, 13, 14, 15, 16, 17, 18, 27, 28, 31, 33, 44, 47, 49, 50, 51, 62, 63, 64, 65, 67, 70, 72, 73, 77, 81, 82, 84, 85, 86, 87, 90]
Final class list: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'train', 'truck', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'backpack', 'umbrella', 'handbag', 'suitcase', 'bottle', 'cup', 'knife', 'spoon', 'bowl', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'cell phone', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'toothbrush', 'crosswalk', 'stairs', 'pothole']


In [ ]:
# ------------------------------------------------------------
# Step 5: Process COCO data (select images with selected classes)
# ------------------------------------------------------------
# Filter annotations
selected_anns = [ann for ann in coco_data['annotations'] if ann['category_id'] in selected_coco_ids]

# Group annotations by image
img_to_anns = defaultdict(list)
for ann in selected_anns:
    img_to_anns[ann['image_id']].append(ann)

# Image info
img_id_to_info = {img['id']: img for img in coco_data['images']}

# For each image, get set of selected classes present
img_to_classes = {img_id: set(ann['category_id'] for ann in anns) for img_id, anns in img_to_anns.items()}

# Collect images per class, up to MAX_IMAGES
class_to_images = defaultdict(list)
for img_id, class_set in img_to_classes.items():
    for cat_id in class_set:
        class_to_images[cat_id].append(img_id)

selected_images = set()
for cat_id, img_list in class_to_images.items():
    if len(img_list) > MAX_IMAGES_PER_COCO_CLASS:
        img_list = random.sample(img_list, MAX_IMAGES_PER_COCO_CLASS)
    selected_images.update(img_list)

print(f"Selected {len(selected_images)} COCO images.")

# Temporary directory for COCO processed data
COCO_TEMP_DIR = os.path.join(BASE_DIR, "temp_coco")
os.makedirs(COCO_TEMP_DIR, exist_ok=True)

for img_id in tqdm(selected_images, desc="Processing COCO images"):
    img_info = img_id_to_info[img_id]
    img_filename = img_info['file_name']
    # Find full path from mapping
    full_img_path = filename_to_path.get(img_filename)
    if full_img_path is None:
        print(f"Warning: Image {img_filename} not found in COCO images directory. Skipping.")
        continue

    # Copy image to temp folder
    dst_img_path = os.path.join(COCO_TEMP_DIR, img_filename)
    shutil.copy(full_img_path, dst_img_path)

    # Convert annotations to YOLO format
    anns = img_to_anns[img_id]
    lines = []
    for ann in anns:
        cat_id = ann['category_id']
        cat_name = coco_cat_id_to_name[cat_id]
        class_idx = class_id_map[cat_name]
        x, y, w, h = ann['bbox']
        img_w = img_info['width']
        img_h = img_info['height']
        x_center = (x + w/2) / img_w
        y_center = (y + h/2) / img_h
        w_norm = w / img_w
        h_norm = h / img_h
        lines.append(f"{class_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

    # Write label
    label_filename = os.path.splitext(img_filename)[0] + '.txt'
    label_path = os.path.join(COCO_TEMP_DIR, label_filename)
    with open(label_path, 'w') as f:
        f.write('\n'.join(lines))


Selected 21327 COCO images.


Processing COCO images:   0%|          | 55/21327 [00:24<2:28:14,  2.39it/s]

Processing COCO images:   0%|          | 96/21327 [00:40<2:29:46,  2.36it/s]

Processing COCO images:   0%|          | 100/21327 [00:42<2:04:30,  2.84it/s]

Processing COCO images:   1%|          | 117/21327 [01:04<2:42:19,  2.18it/s]

...

Processing COCO images:  98%|█████████▊| 20821/21327 [22:09<00:07, 69.96it/s]

Processing COCO images:  99%|█████████▊| 21055/21327 [22:09<00:01, 206.52it/s]

Processing COCO images:  99%|█████████▉| 21084/21327 [22:10<00:01, 164.57it/s]

Processing COCO images:  99%|█████████▉| 21101/21327 [22:10<00:01, 140.48it/s]

Processing COCO images: 100%|██████████| 21327/21327 [22:10<00:00, 16.03it/s] 

In [ ]:

# ------------------------------------------------------------
# Step 6: Process custom data
# ------------------------------------------------------------
CUSTOM_TEMP_DIR = os.path.join(BASE_DIR, "temp_custom")
os.makedirs(CUSTOM_TEMP_DIR, exist_ok=True)

for class_name in CUSTOM_CLASSES:
    images_dir = os.path.join(CUSTOM_ROOT, class_name, 'images')
    labels_dir = os.path.join(CUSTOM_ROOT, class_name, 'labels')

    if not os.path.exists(images_dir) or not os.path.exists(labels_dir):
        print(f"Warning: Missing images or labels for {class_name}. Skipping.")
        continue

    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    for img_file in tqdm(image_files, desc=f"Processing {class_name}"):
        src_img = os.path.join(images_dir, img_file)
        label_file = os.path.splitext(img_file)[0] + '.txt'
        src_label = os.path.join(labels_dir, label_file)

        if not os.path.exists(src_label):
            print(f"Warning: No label for {img_file}, skipping.")
            continue

        # Read original label and update class IDs
        with open(src_label, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            orig_id = int(parts[0])
            if orig_id == CUSTOM_ORIGINAL_IDS[class_name]:
                new_id = class_id_map[class_name]
                parts[0] = str(new_id)
                new_lines.append(' '.join(parts))
            else:
                # Unexpected – ignore or warn
                print(f"Unexpected class id {orig_id} in {class_name} label {label_file}")

        # Write updated label to temp folder
        dst_label = os.path.join(CUSTOM_TEMP_DIR, label_file)
        with open(dst_label, 'w') as f:
            f.write('\n'.join(new_lines))

        # Copy image
        dst_img = os.path.join(CUSTOM_TEMP_DIR, img_file)
        shutil.copy(src_img, dst_img)


Processing crosswalk:   0%|          | 3/800 [00:03<14:26,  1.09s/it]

Processing crosswalk:   1%|          | 6/800 [00:05<10:11,  1.30it/s]

Processing crosswalk:   1%|          | 9/800 [00:05<06:24,  2.06it/s]

Processing crosswalk:   2%|▏         | 13/800 [00:18<36:28,  2.78s/it]

Processing crosswalk:   3%|▎         | 22/800 [00:20<10:21,  1.25it/s]

...

Processing crosswalk:  40%|████      | 321/800 [01:16<01:41,  4.74it/s]

Processing crosswalk:  40%|████      | 324/800 [01:17<02:06,  3.77it/s]

Processing crosswalk:  41%|████      | 328/800 [01:18<02:37,  3.00it/s]

Processing crosswalk: 100%|██████████| 800/800 [01:19<00:00, 10.10it/s]


Processing pothole: 100%|██████████| 665/665 [05:00<00:00,  2.21it/s]


In [ ]:
# ------------------------------------------------------------
# Step 7: Merge and split into train/val (80/20)
# ------------------------------------------------------------
# Gather all files from both temp folders
coco_files = [f for f in os.listdir(COCO_TEMP_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
custom_files = [f for f in os.listdir(CUSTOM_TEMP_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"COCO images: {len(coco_files)}")
print(f"Custom images: {len(custom_files)}")

def copy_with_prefix(src_dir, filename, dst_img_dir, dst_lbl_dir, prefix=''):
    src_img = os.path.join(src_dir, filename)
    dst_img_name = prefix + filename if prefix else filename
    dst_img = os.path.join(dst_img_dir, dst_img_name)
    shutil.copy(src_img, dst_img)

    label_file = os.path.splitext(filename)[0] + '.txt'
    src_label = os.path.join(src_dir, label_file)
    if os.path.exists(src_label):
        dst_label_name = prefix + label_file if prefix else label_file
        dst_label = os.path.join(dst_lbl_dir, dst_label_name)
        shutil.copy(src_label, dst_label)
    else:
        print(f"Warning: No label for {filename}")

# Split COCO
random.shuffle(coco_files)
split_idx = int(len(coco_files) * 0.8)
train_coco = coco_files[:split_idx]
val_coco = coco_files[split_idx:]

for f in tqdm(train_coco, desc="COCO train"):
    copy_with_prefix(COCO_TEMP_DIR, f,
                     os.path.join(OUTPUT_DATASET_DIR, 'images', 'train'),
                     os.path.join(OUTPUT_DATASET_DIR, 'labels', 'train'))

for f in tqdm(val_coco, desc="COCO val"):
    copy_with_prefix(COCO_TEMP_DIR, f,
                     os.path.join(OUTPUT_DATASET_DIR, 'images', 'val'),
                     os.path.join(OUTPUT_DATASET_DIR, 'labels', 'val'))

# Split custom with a prefix to avoid collisions
prefix = "custom_"
random.shuffle(custom_files)
split_idx = int(len(custom_files) * 0.8)
train_custom = custom_files[:split_idx]
val_custom = custom_files[split_idx:]

for f in tqdm(train_custom, desc="Custom train"):
    copy_with_prefix(CUSTOM_TEMP_DIR, f,
                     os.path.join(OUTPUT_DATASET_DIR, 'images', 'train'),
                     os.path.join(OUTPUT_DATASET_DIR, 'labels', 'train'),
                     prefix=prefix)

for f in tqdm(val_custom, desc="Custom val"):
    copy_with_prefix(CUSTOM_TEMP_DIR, f,
                     os.path.join(OUTPUT_DATASET_DIR, 'images', 'val'),
                     os.path.join(OUTPUT_DATASET_DIR, 'labels', 'val'),
                     prefix=prefix)


COCO images: 3191
Custom images: 919


Custom val: 100%|██████████| 184/184 [00:04<00:00, 38.40it/s]


In [ ]:
# ------------------------------------------------------------
# Step 8: Create data.yaml
# ------------------------------------------------------------
data_yaml = {
    'train': os.path.join(OUTPUT_DATASET_DIR, 'images', 'train'),
    'val': os.path.join(OUTPUT_DATASET_DIR, 'images', 'val'),
    'nc': len(final_classes),
    'names': final_classes
}

data_yaml_path = os.path.join(OUTPUT_DATASET_DIR, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml created at: {data_yaml_path}")



data.yaml created at: /content/drive/MyDrive/final_yolo_retraining (1)/yolo_dataset/data.yaml


In [ ]:
import os
import yaml
import torch

# Clear CUDA cache to fix PyTorch issues
torch.cuda.empty_cache()

# Define the correct base directory (the one with your actual data)
BASE_DIR = "/content/drive/MyDrive/final_yolo_retraining"
DATASET_DIR = os.path.join(BASE_DIR, "yolo_dataset")
TRAINING_OUTPUT_DIR = os.path.join(BASE_DIR, "runs")

# Define your classes (make sure these match what you have)
SELECTED_COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'train', 'truck',
    'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'backpack', 'umbrella', 'handbag', 'suitcase',
    'bottle', 'cup', 'knife', 'spoon', 'bowl', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop',
    'cell phone', 'sink', 'refrigerator', 'book', 'clock', 'vase',
    'scissors', 'toothbrush'
]

CUSTOM_CLASSES = ['crosswalk', 'stairs', 'pothole']
final_classes = SELECTED_COCO_CLASSES + CUSTOM_CLASSES

print(f"Total classes: {len(final_classes)}")

# Create data.yaml with absolute paths
data_yaml = {
    'train': os.path.join(DATASET_DIR, 'images', 'train'),
    'val': os.path.join(DATASET_DIR, 'images', 'val'),
    'nc': len(final_classes),
    'names': final_classes
}

data_yaml_path = os.path.join(DATASET_DIR, 'data.yaml')

# Write the file
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"✅ data.yaml created at: {data_yaml_path}")

# Verify the paths
print(f"\n📂 Verifying paths:")
print(f"   Train: {data_yaml['train']}")
print(f"   Train exists: {os.path.exists(data_yaml['train'])}")
print(f"   Val: {data_yaml['val']}")
print(f"   Val exists: {os.path.exists(data_yaml['val'])}")

# Check how many images in each split
if os.path.exists(data_yaml['train']):
    train_images = [f for f in os.listdir(data_yaml['train']) if f.endswith(('.jpg', '.jpeg', '.png'))]
    print(f"   Train images count: {len(train_images)}")
if os.path.exists(data_yaml['val']):
    val_images = [f for f in os.listdir(data_yaml['val']) if f.endswith(('.jpg', '.jpeg', '.png'))]
    print(f"   Val images count: {len(val_images)}")

Total classes: 43
✅ data.yaml created at: /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/data.yaml

📂 Verifying paths:
   Train: /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/images/train
   Train exists: True
   Val: /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/images/val
   Val exists: True
   Train images count: 3287
   Val images count: 823


In [ ]:
# First, clear all caches and restart
import torch
import os
import shutil

# Clear PyTorch cache
torch.cuda.empty_cache()

# Remove any corrupted model files
cache_dir = os.path.expanduser('~/.cache/ultralytics')
if os.path.exists(cache_dir):
    print(f"Removing corrupted cache: {cache_dir}")
    shutil.rmtree(cache_dir)
    print("✅ Cache cleared")

# Also check for any local .pt files that might be corrupted
pt_files = [f for f in os.listdir('.') if f.endswith('.pt')]
for pt_file in pt_files:
    print(f"Removing local {pt_file}")
    os.remove(pt_file)

print("\n✅ Cleanup complete. Now reinstalling ultralytics...")

# Reinstall ultralytics
!pip uninstall ultralytics -y
!pip install ultralytics -q

# Verify installation
import ultralytics
print(f"✅ Ultralytics version: {ultralytics.__version__}")

Removing local yolov8m.pt

✅ Cleanup complete. Now reinstalling ultralytics...
Found existing installation: ultralytics 8.4.30
Uninstalling ultralytics-8.4.30:
  Successfully uninstalled ultralytics-8.4.30
✅ Ultralytics version: 8.4.30


In [ ]:
from ultralytics import YOLO
import torch

# Clear cache again
torch.cuda.empty_cache()

# Download model fresh (this will download from official source)
print("Downloading YOLOv8m model...")
try:
    model = YOLO('yolov8m.pt')
    print("✅ Model downloaded and loaded successfully!")

    # Test if model works
    print(f"Model type: {type(model)}")
    print(f"Model info: {model.model}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\nTrying alternative method...")

    # Alternative: download manually
    import urllib.request
    url = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8m.pt"
    urllib.request.urlretrieve(url, "yolov8m.pt")
    model = YOLO("yolov8m.pt")
    print("✅ Model downloaded manually and loaded!")

✅ Model downloaded and loaded successfully!
Model type: <class 'ultralytics.models.yolo.model.YOLO'>
Model info: DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(9

In [ ]:
# Step 1: Mount Drive and Setup
from google.colab import drive
drive.mount('/content/drive')

import os
import yaml
import torch

# Force CPU usage
device = 'cpu'
print(f"Using device: {device}")

# Define paths
BASE_DIR = "/content/drive/MyDrive/final_yolo_retraining"
DATASET_DIR = os.path.join(BASE_DIR, "yolo_dataset")
TRAINING_OUTPUT_DIR = os.path.join(BASE_DIR, "runs")
data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")

# Define classes
SELECTED_COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'train', 'truck',
    'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'backpack', 'umbrella', 'handbag', 'suitcase',
    'bottle', 'cup', 'knife', 'spoon', 'bowl', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop',
    'cell phone', 'sink', 'refrigerator', 'book', 'clock', 'vase',
    'scissors', 'toothbrush'
]
CUSTOM_CLASSES = ['crosswalk', 'stairs', 'pothole']
final_classes = SELECTED_COCO_CLASSES + CUSTOM_CLASSES

# Create or verify data.yaml
if not os.path.exists(data_yaml_path):
    data_yaml = {
        'train': os.path.join(DATASET_DIR, 'images', 'train'),
        'val': os.path.join(DATASET_DIR, 'images', 'val'),
        'nc': len(final_classes),
        'names': final_classes
    }
    with open(data_yaml_path, 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
    print(f"✅ Created data.yaml at {data_yaml_path}")

# Verify dataset
print(f"\n📊 Dataset Verification:")
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
print(f"   Classes: {data_config['nc']}")
print(f"   Train path: {data_config['train']}")
print(f"   Train exists: {os.path.exists(data_config['train'])}")
print(f"   Val path: {data_config['val']}")
print(f"   Val exists: {os.path.exists(data_config['val'])}")

if os.path.exists(data_config['train']):
    train_count = len([f for f in os.listdir(data_config['train']) if f.endswith(('.jpg', '.jpeg', '.png'))])
    print(f"   Train images: {train_count}")
if os.path.exists(data_config['val']):
    val_count = len([f for f in os.listdir(data_config['val']) if f.endswith(('.jpg', '.jpeg', '.png'))])
    print(f"   Val images: {val_count}")

# Install ultralytics
!pip install ultralytics -q

from ultralytics import YOLO

# Training parameters (adjust batch size for CPU)
EPOCHS = 50  # Start with fewer epochs since CPU is slower
BATCH_SIZE = 4  # Smaller batch size for CPU
IMAGE_SIZE = 416  # Smaller image size for faster training
MODEL_SIZE = 'n'  # Use nano model for CPU training
SAVE_PERIOD = 5

print(f"\n{'='*60}")
print(f"🚀 Starting Training on CPU")
print(f"{'='*60}")
print(f"   Model: yolov8{MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Image size: {IMAGE_SIZE}")
print(f"   Save period: {SAVE_PERIOD}")
print(f"   Device: CPU (GPU limit reached)")
print(f"{'='*60}")
print(f"\n⚠️  Note: CPU training is slower. Expect ~5-10 minutes per epoch.")
print(f"   Consider training overnight or in smaller chunks.\n")

# Load model
print("Loading YOLOv8 model...")
model = YOLO(f'yolov8{MODEL_SIZE}.pt')
print("✅ Model loaded successfully\n")

# Train on CPU
results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    name='custom_yolov8_cpu',
    project=TRAINING_OUTPUT_DIR,
    save_period=SAVE_PERIOD,
    patience=10,
    device='cpu',  # Force CPU usage
    verbose=True
)

print("\n✅ Training completed!")

# Evaluate
print(f"\n{'='*60}")
print("📊 Evaluating best model...")
print(f"{'='*60}")

best_model_path = os.path.join(TRAINING_OUTPUT_DIR, 'train', 'custom_yolov8_cpu', 'weights', 'best.pt')
if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    metrics = best_model.val(data=data_yaml_path, split='val', device='cpu')

    print(f"\n📈 Overall Metrics:")
    print(f"   - mAP50: {metrics.box.map50:.4f}")
    print(f"   - mAP50-95: {metrics.box.map:.4f}")

    # Show custom classes performance
    coco_count = len(SELECTED_COCO_CLASSES)
    print(f"\n🎯 Custom Classes Performance:")
    for i, class_name in enumerate(CUSTOM_CLASSES):
        idx = coco_count + i
        if idx < len(metrics.box.ap50):
            print(f"   - {class_name}: mAP50 = {metrics.box.ap50[idx]:.4f}")
else:
    print(f"⚠️ Best model not found")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu

📊 Dataset Verification:
   Classes: 43
   Train path: /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/images/train
   Train exists: True
   Val path: /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/images/val
   Val exists: True
   Train images: 3287
   Val images: 823

🚀 Starting Training on CPU
   Model: yolov8n
   Epochs: 50
   Batch size: 4
   Image size: 416
   Save period: 5
   Device: CPU (GPU limit reached)

⚠️  Note: CPU training is slower. Expect ~5-10 minutes per epoch.
   Consider training overnight or in smaller chunks.

Loading YOLOv8 model...
✅ Model loaded successfully

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes

In [ ]:
# ------------------------------------------------------------
# Step 10: Evaluate
# ------------------------------------------------------------
best_model_path = os.path.join(TRAINING_OUTPUT_DIR, 'train', 'custom_yolov8', 'weights', 'best.pt')
best_model = YOLO(best_model_path)

metrics = best_model.val(data=data_yaml_path, split='val')
print(f"Overall mAP50: {metrics.box.map50:.4f}")
print(f"Overall mAP50-95: {metrics.box.map:.4f}")
print("Per-class mAP50:")
for i, name in enumerate(final_classes):
    print(f"  {name}: {metrics.box.ap50[i]:.4f}")

RE RUN

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ============================================================================
# DATASET VERIFICATION – RUN THIS BEFORE TRAINING
# ============================================================================

import os
import yaml
from collections import defaultdict
import random

# Paths (adjust if needed)
BASE_DIR = "/content/drive/MyDrive/final_yolo_retraining"
DATASET_DIR = os.path.join(BASE_DIR, "yolo_dataset")
data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")

print("=" * 70)
print("🔍 YOLO DATASET VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------------
# 1. Check data.yaml exists and is valid
# ------------------------------------------------------------------
if not os.path.exists(data_yaml_path):
    print(f"❌ ERROR: data.yaml not found at {data_yaml_path}")
    print("   Please create the dataset first.")
    exit(1)
else:
    print(f"✅ data.yaml found: {data_yaml_path}")

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

required_keys = ['train', 'val', 'nc', 'names']
for key in required_keys:
    if key not in data_config:
        print(f"❌ ERROR: '{key}' missing in data.yaml")
        exit(1)

print(f"   - Number of classes (nc): {data_config['nc']}")
print(f"   - Class names: {data_config['names'][:5]}... (showing first 5)")

# ------------------------------------------------------------------
# 2. Check train and val directories
# ------------------------------------------------------------------
train_img_dir = data_config['train']
val_img_dir = data_config['val']

if not os.path.exists(train_img_dir):
    print(f"❌ ERROR: Train image directory not found: {train_img_dir}")
    exit(1)
if not os.path.exists(val_img_dir):
    print(f"❌ ERROR: Validation image directory not found: {val_img_dir}")
    exit(1)

print(f"\n📁 Train directory: {train_img_dir}")
print(f"📁 Val directory:   {val_img_dir}")

# ------------------------------------------------------------------
# 3. Count images and labels
# ------------------------------------------------------------------
def count_files(dir_path, extensions):
    files = []
    for ext in extensions:
        files.extend([f for f in os.listdir(dir_path) if f.lower().endswith(ext)])
    return files

train_images = count_files(train_img_dir, ('.jpg', '.jpeg', '.png'))
train_labels_dir = train_img_dir.replace('images', 'labels')
train_labels = count_files(train_labels_dir, ('.txt',)) if os.path.exists(train_labels_dir) else []

val_images = count_files(val_img_dir, ('.jpg', '.jpeg', '.png'))
val_labels_dir = val_img_dir.replace('images', 'labels')
val_labels = count_files(val_labels_dir, ('.txt',)) if os.path.exists(val_labels_dir) else []

print(f"\n📊 Training set:")
print(f"   - Images: {len(train_images)}")
print(f"   - Labels: {len(train_labels)}")
if len(train_images) != len(train_labels):
    print(f"   ⚠️  WARNING: Image count ({len(train_images)}) ≠ Label count ({len(train_labels)})")

print(f"\n📊 Validation set:")
print(f"   - Images: {len(val_images)}")
print(f"   - Labels: {len(val_labels)}")
if len(val_images) != len(val_labels):
    print(f"   ⚠️  WARNING: Image count ({len(val_images)}) ≠ Label count ({len(val_labels)})")

# ------------------------------------------------------------------
# 4. Verify label format and class IDs
# ------------------------------------------------------------------
def check_labels(label_dir, max_class_id, split_name):
    if not os.path.exists(label_dir):
        print(f"   ⚠️  {split_name}: Label directory missing: {label_dir}")
        return True

    class_ids_found = set()
    invalid_lines = 0
    total_labels = 0
    empty_files = 0

    for label_file in os.listdir(label_dir):
        if not label_file.endswith('.txt'):
            continue
        file_path = os.path.join(label_dir, label_file)
        total_labels += 1

        with open(file_path, 'r') as f:
            lines = f.readlines()
            if len(lines) == 0:
                empty_files += 1
                continue
            for line in lines:
                parts = line.strip().split()
                if len(parts) != 5:
                    invalid_lines += 1
                    continue
                try:
                    class_id = int(parts[0])
                    if class_id < 0 or class_id > max_class_id:
                        invalid_lines += 1
                    else:
                        class_ids_found.add(class_id)
                    # Check coordinates are in [0,1]
                    for coord in parts[1:]:
                        val = float(coord)
                        if val < 0 or val > 1:
                            invalid_lines += 1
                except ValueError:
                    invalid_lines += 1

    print(f"\n   {split_name} label check:")
    print(f"   - Label files: {total_labels}")
    if empty_files > 0:
        print(f"   - Empty label files: {empty_files} (no objects)")
    if invalid_lines > 0:
        print(f"   - Invalid lines: {invalid_lines}")

    if class_ids_found:
        print(f"   - Class IDs found: {sorted(class_ids_found)}")
        missing_classes = set(range(max_class_id+1)) - class_ids_found
        if missing_classes:
            print(f"   - ⚠️  Classes missing in {split_name}: {sorted(missing_classes)}")
    else:
        print(f"   - No valid class IDs found!")
        return False
    return True

max_class_id = data_config['nc'] - 1
print(f"\n🔢 Expected class IDs: 0 to {max_class_id}")

train_ok = check_labels(train_labels_dir, max_class_id, "Training")
val_ok = check_labels(val_labels_dir, max_class_id, "Validation")

# ------------------------------------------------------------------
# 5. Check for image-label mismatches (orphaned files)
# ------------------------------------------------------------------
def find_orphans(image_dir, label_dir):
    image_names = {os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))}
    label_names = {os.path.splitext(f)[0] for f in os.listdir(label_dir) if f.endswith('.txt')}
    images_without_labels = image_names - label_names
    labels_without_images = label_names - image_names
    return images_without_labels, labels_without_images

train_orphan_imgs, train_orphan_lbls = find_orphans(train_img_dir, train_labels_dir)
val_orphan_imgs, val_orphan_lbls = find_orphans(val_img_dir, val_labels_dir)

if train_orphan_imgs:
    print(f"\n⚠️  Training images without labels: {len(train_orphan_imgs)} (first 5: {list(train_orphan_imgs)[:5]})")
if train_orphan_lbls:
    print(f"⚠️  Training labels without images: {len(train_orphan_lbls)} (first 5: {list(train_orphan_lbls)[:5]})")
if val_orphan_imgs:
    print(f"⚠️  Validation images without labels: {len(val_orphan_imgs)} (first 5: {list(val_orphan_imgs)[:5]})")
if val_orphan_lbls:
    print(f"⚠️  Validation labels without images: {len(val_orphan_lbls)} (first 5: {list(val_orphan_lbls)[:5]})")

# ------------------------------------------------------------------
# 6. Sample a few random images and show their labels
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("📸 SAMPLE CHECK (random images with their labels)")
print("=" * 70)

def show_sample(image_dir, label_dir, split_name, num_samples=3):
    images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    if not images:
        print(f"{split_name}: No images to sample.")
        return
    samples = random.sample(images, min(num_samples, len(images)))
    for img in samples:
        label_file = os.path.splitext(img)[0] + '.txt'
        label_path = os.path.join(label_dir, label_file)
        print(f"\n   {split_name} image: {img}")
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
                for line in lines[:3]:  # show at most 3 lines
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        class_name = data_config['names'][class_id] if class_id < len(data_config['names']) else "UNKNOWN"
                        print(f"      - class {class_id} ({class_name}), box: {parts[1:3]} ...")
                if len(lines) > 3:
                    print(f"      ... and {len(lines)-3} more objects")
        else:
            print(f"      ❌ No label file found!")

if train_images:
    show_sample(train_img_dir, train_labels_dir, "Train")
if val_images:
    show_sample(val_img_dir, val_labels_dir, "Val")

# ------------------------------------------------------------------
# 7. Final verdict
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("📋 VERIFICATION SUMMARY")
print("=" * 70)

all_ok = True
if len(train_images) == 0 or len(val_images) == 0:
    print("❌ CRITICAL: No training or validation images found.")
    all_ok = False
if len(train_labels) == 0 or len(val_labels) == 0:
    print("❌ CRITICAL: No label files found.")
    all_ok = False
if len(train_orphan_imgs) > len(train_images) * 0.1:  # more than 10% orphans
    print("⚠️  Many images lack labels – training may be inefficient.")
if not train_ok or not val_ok:
    print("❌ Label format errors detected. Fix them before training.")
    all_ok = False

if all_ok:
    print("\n✅ ALL CHECKS PASSED! Your dataset is ready for training.")
    print("🚀 You can now run the GPU training script.")
else:
    print("\n❌ Some issues were found. Please fix them before training.")
    print("   Common fixes:")
    print("   - Ensure every image has a corresponding .txt label file")
    print("   - Verify label files have 5 columns: class x_center y_center width height (normalized 0-1)")
    print("   - Check class IDs are within 0..nc-1")
    print("   - Run the dataset creation script again if needed.")

🔍 YOLO DATASET VERIFICATION
✅ data.yaml found: /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/data.yaml
   - Number of classes (nc): 43
   - Class names: ['person', 'bicycle', 'car', 'motorcycle', 'bus']... (showing first 5)

📁 Train directory: /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/images/train
📁 Val directory:   /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/images/val

📊 Training set:
   - Images: 3287
   - Labels: 3287

📊 Validation set:
   - Images: 823
   - Labels: 823

🔢 Expected class IDs: 0 to 42

   Training label check:
   - Label files: 3287
   - Class IDs found: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42]

   Validation label check:
   - Label files: 823
   - Class IDs found: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37,

In [ ]:
# Install Ultralytics (includes YOLOv8 and all dependencies)
!pip install ultralytics

# Optional: Upgrade if already installed (recommended for latest features)
!pip install -U ultralytics
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

from ultralytics import YOLO
print("✅ YOLO imported successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics version: 8.4.33
✅ YOLO imported successfully


In [ ]:
# ============================================================================
# COMPLETE YOLOv8 TRAINING SCRIPT – GPU OPTIMIZED
# Dataset location: /content/drive/MyDrive/final_yolo_retraining/yolo_dataset
# ============================================================================

# 1. Mount Google Drive and set up paths
from google.colab import drive
drive.mount('/content/drive')

import os
import yaml
import torch
import shutil
from ultralytics import YOLO

# ----------------------------------------------------------------------------
# CONFIGURATION – UPDATE THIS IF NEEDED
# ----------------------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/final_yolo_retraining"   # Your main folder
DATASET_DIR = os.path.join(BASE_DIR, "yolo_dataset")        # Where the dataset lives
TRAINING_OUTPUT_DIR = os.path.join(BASE_DIR, "runs")        # Where training logs/weights go
data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")

# Training hyperparameters (optimised for GPU)
MODEL_SIZE = 'm'            # 'n', 's', 'm', 'l', 'x' – m is good balance
EPOCHS = 100                # Total epochs
BATCH_SIZE = 16             # Adjust based on GPU memory (16 works on T4/V100)
IMAGE_SIZE = 640            # Standard YOLO input size
SAVE_PERIOD = 5             # Save a checkpoint every 5 epochs
PATIENCE = 15               # Early stopping if no improvement for 15 epochs
WORKERS = 8                 # Number of data loading threads

# ----------------------------------------------------------------------------
# 2. Verify GPU availability
# ----------------------------------------------------------------------------
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"\n{'='*60}")
print(f"🚀 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("   ⚠️  GPU not found – training will be extremely slow on CPU.")
    print("   Consider restarting Colab with GPU runtime.")
print(f"{'='*60}\n")

# ----------------------------------------------------------------------------
# 3. Ensure data.yaml exists (it does, but just in case)
# ----------------------------------------------------------------------------
if not os.path.exists(data_yaml_path):
    print(f"❌ ERROR: data.yaml not found at {data_yaml_path}")
    print("Please check your dataset directory.")
    exit()
else:
    print(f"✅ data.yaml found: {data_yaml_path}")

# ----------------------------------------------------------------------------
# 4. Validate dataset structure (quick check)
# ----------------------------------------------------------------------------
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

train_img_dir = data_config['train']
val_img_dir = data_config['val']

if not os.path.exists(train_img_dir):
    raise FileNotFoundError(f"Train directory missing: {train_img_dir}")
if not os.path.exists(val_img_dir):
    raise FileNotFoundError(f"Validation directory missing: {val_img_dir}")

train_images = len([f for f in os.listdir(train_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
val_images = len([f for f in os.listdir(val_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
print(f"\n📊 Dataset confirmed:")
print(f"   Training images: {train_images}")
print(f"   Validation images: {val_images}")
print(f"   Number of classes: {data_config['nc']}\n")

# ----------------------------------------------------------------------------
# 5. Check for existing checkpoint to resume
# ----------------------------------------------------------------------------
model_name = 'custom_yolov8_gpu'
weights_dir = os.path.join(TRAINING_OUTPUT_DIR, 'train', model_name, 'weights')
last_checkpoint = os.path.join(weights_dir, 'last.pt')
resume_training = os.path.exists(last_checkpoint)

if resume_training:
    print(f"🔄 Found existing checkpoint: {last_checkpoint}")
    print("   Training will resume from the last saved epoch.\n")
else:
    print("✨ No previous checkpoint found. Starting fresh training.\n")

# ----------------------------------------------------------------------------
# 6. Train the model (with resuming, checkpoints, and early stopping)
# ----------------------------------------------------------------------------
print(f"{'='*60}")
print(f"🚀 STARTING TRAINING")
print(f"{'='*60}")
print(f"   Model: yolov8{MODEL_SIZE}.pt")
print(f"   Device: {device}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Image size: {IMAGE_SIZE}")
print(f"   Save checkpoint every: {SAVE_PERIOD} epochs")
print(f"   Early stopping patience: {PATIENCE}")
print(f"{'='*60}\n")

# Load model (from checkpoint if resuming)
if resume_training:
    model = YOLO(last_checkpoint)
else:
    model = YOLO(f'yolov8{MODEL_SIZE}.pt')

# Train with all optimisations
results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    name=model_name,
    project=TRAINING_OUTPUT_DIR,
    save_period=SAVE_PERIOD,
    patience=PATIENCE,          # Early stopping
    device=device,              # Use GPU if available
    resume=resume_training,     # Automatically resume from last checkpoint
    exist_ok=True,              # Allow overwriting existing run
    verbose=True,
    # Optimisation flags
    amp=True,                   # Automatic Mixed Precision (faster training)
    seed=42,                    # Reproducibility
    deterministic=True,         # CuDNN deterministic mode
)

print("\n✅ Training completed!")

# ----------------------------------------------------------------------------
# 7. Evaluate the best model
# ----------------------------------------------------------------------------
best_model_path = os.path.join(weights_dir, 'best.pt')
if os.path.exists(best_model_path):
    print(f"\n{'='*60}")
    print("📊 EVALUATING BEST MODEL")
    print(f"{'='*60}")
    best_model = YOLO(best_model_path)
    metrics = best_model.val(data=data_yaml_path, split='val', device=device)

    print(f"\n📈 Overall Metrics:")
    print(f"   - mAP50: {metrics.box.map50:.4f}")
    print(f"   - mAP50-95: {metrics.box.map:.4f}")

    # Per‑class performance for custom classes (IDs 40,41,42)
    # Based on your class list: last three are crosswalk, stairs, pothole
    custom_class_names = ['crosswalk', 'stairs', 'pothole']
    custom_start_idx = data_config['nc'] - 3  # Should be 40
    print(f"\n🎯 Custom Classes Performance:")
    for i, class_name in enumerate(custom_class_names):
        idx = custom_start_idx + i
        if idx < len(metrics.box.ap50):
            print(f"   - {class_name}: mAP50 = {metrics.box.ap50[idx]:.4f}")
        else:
            print(f"   - {class_name}: metric not available")
else:
    print(f"\n⚠️  Best model not found at {best_model_path}")

# ----------------------------------------------------------------------------
# 8. Save a final backup (just in case)
# ----------------------------------------------------------------------------
final_backup = os.path.join(weights_dir, 'final_backup.pt')
if os.path.exists(last_checkpoint):
    shutil.copy(last_checkpoint, final_backup)
    print(f"\n💾 Final backup saved to: {final_backup}")

print("\n🎉 All done! Your model is ready for inference.")
print(f"Best model location: {best_model_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 Device: cuda:0
   GPU name: Tesla T4
   GPU memory: 15.64 GB

✅ data.yaml found: /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/data.yaml

📊 Dataset confirmed:
   Training images: 3287
   Validation images: 823
   Number of classes: 43

✨ No previous checkpoint found. Starting fresh training.

🚀 STARTING TRAINING
   Model: yolov8m.pt
   Device: cuda:0
   Epochs: 100
   Batch size: 16
   Image size: 640
   Save checkpoint every: 5 epochs
   Early stopping patience: 15

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/My

In [ ]:
from ultralytics import YOLO

best_model_path = "/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.pt"
model = YOLO(best_model_path)
metrics = model.val(data="/content/drive/MyDrive/final_yolo_retraining/yolo_dataset/data.yaml")

print(f"Overall mAP50: {metrics.box.map50:.4f}")

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,864,657 parameters, 0 gradients, 78.8 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 54.8±22.3 MB/s, size: 136.5 KB)
val: Scanning /content/drive/MyDrive/final_yolo_retraining/yolo_dataset/labels/val.cache... 823 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 823/823 246.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 1.8it/s 28.2s
                   all        823       5803      0.599      0.548      0.571      0.408
                person        349       1469      0.626      0.796      0.782      0.553
               bicycle         44         87      0.644      0.437      0.512      0.314
                   car        112        416      0.543       0.57      0.475      0.307
            motorcycle         28         42      0.421      0.595      0.568      0.377
        

Downloading the model

In [ ]:
import shutil
cache_dir = '/root/.cache/ultralytics'
shutil.rmtree(cache_dir, ignore_errors=True)

In [ ]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.7 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade pip
!pip install ultralytics tensorflow onnx onnx2tf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
from ultralytics import YOLO

best_model_path = "/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.pt"
model = YOLO(best_model_path)

In [ ]:
# Export to TFLite
model.export(format='tflite', imgsz=640)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
Model summary (fused): 93 layers, 25,864,657 parameters, 0 gradients, 78.8 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 47, 8400) (49.6 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 4.8s, saved as '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.onnx' (99.0 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Ty

'/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best_saved_model/best_float32.tflite'

In [ ]:
from ultralytics import YOLO

best_model_path = "/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.pt"
model = YOLO(best_model_path)

# Export TFLite and save in current folder
model.export(format='tflite', imgsz=640, opset=19, save_dir="/content")

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
Model summary (fused): 93 layers, 25,864,657 parameters, 0 gradients, 78.8 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 47, 8400) (49.6 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.21.0 opset 19...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 3.8s, saved as '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.onnx' (99.0 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Ty

'/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best_saved_model/best_float32.tflite'

In [ ]:
!ls /content

calibration_image_sample_data_20x128x128x3_float32.npy	drive  sample_data


In [ ]:
from ultralytics import YOLO

best_model_path = "/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.pt"
model = YOLO(best_model_path)

# Export TFLite and save in /content for easy download
model.export(format='tflite', imgsz=640, save_dir="/content")

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
Model summary (fused): 93 layers, 25,864,657 parameters, 0 gradients, 78.8 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 47, 8400) (49.6 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 3.1s, saved as '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.onnx' (99.0 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Ty

'/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best_saved_model/best_float32.tflite'

In [ ]:
from google.colab import files
files.download("/content/yolov8n.tflite")

FileNotFoundError: Cannot find file: /content/yolov8n.tflite

In [ ]:
!pip install --upgrade pip
!pip install ultralytics tensorflow onnx onnx2tf onnxsim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.0 MB/s  0:00:00


In [ ]:
from ultralytics import YOLO

best_model_path = "/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.pt"
model = YOLO(best_model_path)

# Export TFLite to /content
export_path = model.export(format='tflite', imgsz=640, save_dir="/content")
print(f"TFLite exported to: {export_path}")

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
Model summary (fused): 93 layers, 25,864,657 parameters, 0 gradients, 78.8 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 47, 8400) (49.6 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 3.8s, saved as '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best.onnx' (99.0 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/final_yolo_retraining/runs/custom_yolov8_gpu/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Ty

In [ ]:
from google.colab import files
files.download(export_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Step 1: Install YOLOv8
!pip install ultralytics

# Step 2: Import YOLO and load pretrained model
from ultralytics import YOLO

# You can choose yolov8n, yolov8s, etc. 'n' = nano (smallest)
model = YOLO('yolov8n.pt')

# Step 3: Export to TFLite
# 'simplify=True' can help reduce model size
model.export(format='tflite', simplify=True)

from google.colab import files
files.download('yolov8n_saved_model/yolov8n_float32.tflite')

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 1.1s, saved as 'yolov8n.onnx' (12.4 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at 'yolov8n_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 84, 8400), dtype=tf.float32, name=None)
Captures:
  132894505530192: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  132894505517904: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, nam

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1️⃣ Install dependencies
!pip install torch torchvision timm opencv-python --quiet

# 2️⃣ Download MiDaS small model directly
!mkdir -p weights
!wget https://github.com/isl-org/MiDaS/releases/download/v3_1/midas_v21_small.pt -O weights/midas_v21_small.pt

# 3️⃣ Clone latest MiDaS repo
!git clone https://github.com/isl-org/MiDaS.git
%cd MiDaS

# 4️⃣ Install required packages (skip if requirements.txt missing)
!pip install -r requirements.txt || echo "No requirements.txt, skipping"

# 5️⃣ Use updated import from MiDaS v3
import torch
from midas.midas_net_custom import MidasNet_small  # updated path
from midas.transforms import Resize, NormalizeImage, PrepareForNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load MiDaS small model
model_path = "../weights/midas_v21_small.pt"
midas = MidasNet_small(model_path, features=64, backbone="efficientnet_lite3")
midas.to(device)
midas.eval()

print("✅ MiDaS small model loaded successfully!")

# 6️⃣ Optional: download to local
from google.colab import files
files.download("../weights/midas_v21_small.pt")

--2026-04-02 20:07:41--  https://github.com/isl-org/MiDaS/releases/download/v3_1/midas_v21_small.pt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-02 20:07:41 ERROR 404: Not Found.

Cloning into 'MiDaS'...
remote: Enumerating objects: 622, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 622 (delta 183), reused 136 (delta 136), pack-reused 377 (from 2)
Receiving objects: 100% (622/622), 3.44 MiB | 31.16 MiB/s, done.
Resolving deltas: 100% (246/246), done.
/content/MiDaS/MiDaS/MiDaS/MiDaS
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
No requirements.txt, skipping
Loading weights:  ../weights/midas_v21_small.pt
Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip


EOFError: 